In [105]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix
import pickle 
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random
from sklearn.neighbors import NearestNeighbors


In [106]:
# fetching data 
music_info = pd.read_csv('Data/music_info_cleaned.csv')
users_history = pd.read_csv('Data/users_history_cleaned.csv')
music_info_metadata = pd.read_csv('Data/music_info_metadata.csv')

In [107]:
train, test = train_test_split(users_history, test_size=0.2, random_state=42)

#from hybrid model
# For collaborative filtering (using users_history)
train['user_id'] = train['user_id'].astype('category')
train['track_id'] = train['track_id'].astype('category')

# Create mappings for user_id and track_id
cf_user_id_mapping = dict(enumerate(train['user_id'].cat.categories))
cf_track_id_mapping = dict(enumerate(train['track_id'].cat.categories))
cf_user_id_reverse_mapping = {v: k for k, v in cf_user_id_mapping.items()}
cf_track_id_reverse_mapping = {v: k for k, v in cf_track_id_mapping.items()}

# For content-based filtering (using music_info)
music_info['track_id'] = music_info['track_id'].astype('category')

cb_track_id_mapping = dict(enumerate(music_info['track_id'].cat.categories))
cb_track_id_reverse_mapping = {v: k for k, v in cb_track_id_mapping.items()}

print(f"Trening (interaksjoner): {len(train)}")
print(f"Test (interaksjoner): {len(test)}")

Trening (interaksjoner): 1404017
Test (interaksjoner): 351005


In [108]:
# making sparse user item matix 
user_item_sparse = coo_matrix((
    train['playcount'],
    (train['user_id'].cat.codes,
     train['track_id'].cat.codes)
))
# training truncated svd 

SVD_COMPONENTS = 10 

svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
user_factors = svd.fit_transform(user_item_sparse)
item_factors = svd.components_.T 

print(f"SVD-trening fullført.")
print(f"User Factors shape: {user_factors.shape}")
print(f"Item Factors shape: {item_factors.shape}")

SVD-trening fullført.
User Factors shape: (22733, 10)
Item Factors shape: (25903, 10)


In [109]:
# random popularity model

all_items = list(music_info_metadata['track_id'].unique())
track_popularity = train.groupby('track_id')['playcount'].sum().sort_values(ascending=False)
most_popular_songs = track_popularity.head(10).index.tolist()

/var/folders/kw/2mf6m7dx0pn9xqtnjjs396700000gn/T/ipykernel_25030/4101603300.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  track_popularity = train.groupby('track_id')['playcount'].sum().sort_values(ascending=False)


In [110]:
# same as in hybrid model
# Features to be used to compute similarity between traks using kNN
feature_columns = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 
                   'instrumentalness', 'liveness', 'valence', 'tempo']
numerical_features = music_info[feature_columns].values
knn_index = NearestNeighbors(n_neighbors=50, metric='cosine').fit(numerical_features)

In [111]:
user_test_data = test.groupby('user_id')['track_id'].apply(list).to_dict()

In [112]:
# also from hybrid reccomender
# Function to get nearest neighbors for a given track
def get_knn(knn_index, feature_matrix, item_id, n):
    distances, indices = knn_index.kneighbors(numerical_features[item_id:item_id+1], n_neighbors=n+1)
    # Return so it skips the first one (itself)
    return indices[0][1:], distances[0][1:]


In [113]:
# from our hybrid model
def recommend_songs_hybrid(user_id, track_name, user_item_matrix, user_factors, item_factors,
                           music_info_metadata, knn_index, numerical_features, user_mood=None,
                           n_recommendations=5):

    # Using metadata from earlier containing artist and name of track
    track_metadata_dict = music_info_metadata.set_index('track_id')[['name', 'artist']].to_dict('index')
    
    # Validate user
    user_code = cf_user_id_reverse_mapping.get(user_id)
    if user_code is None:
        print(f"User ID {user_id} not found in the user-item matrix.")
        return []

    # Validate track
    track_row = music_info_metadata[music_info_metadata['name'] == track_name]
    if track_row.empty:
        print(f"Track '{track_name}' not found in metadata.")
        return []
        
    track_id = track_row.iloc[0]['track_id']
    track_code = cb_track_id_reverse_mapping.get(track_id)
    if track_code is None:
        print(f"Track '{track_name}' not found.")
        return [], []

    # -------------------
    # Content-Based Filtering with kNN
    # -------------------
    similar_indices, distances_from_knn = get_knn(knn_index, numerical_features, track_code, n=25)
    cb_candidates = music_info_metadata.iloc[similar_indices][['track_id', 'name', 'artist']].copy()
    cb_recommended_tracks = cb_candidates['track_id'].tolist()

    # CB predicted scores (from kNN distances)
    cb_scores = {t: 1 - d for t, d in zip(cb_recommended_tracks, distances_from_knn)}

    # -------------------
    # Collaborative Filtering 
    # -------------------
    cf_predictions = np.dot(user_factors[user_code, :], item_factors.T)
    cf_indices = np.argsort(cf_predictions)[::-1]
    cf_recommended_tracks = [cf_track_id_mapping[i] for i in cf_indices[:n_recommendations]]
    
    # CF predicted scores
    cf_scores = {
        t: np.dot(user_factors[user_code, :], item_factors[cf_track_id_reverse_mapping.get(t, 0), :])
        for t in cf_recommended_tracks
    }

    # Normalizating score to ensure one does not dominate the other
    print("CF score range BEFORE normalization:", min(cf_scores.values()), max(cf_scores.values()))
    print("CB score range BEFORE normalization:", min(cb_scores.values()), max(cb_scores.values()))
    
    def min_max_normalize(scores):
        vals = np.array(list(scores.values()))
        if vals.max() == vals.min():
            return {k: 0.5 for k in scores}  # fallback if all scores equal
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in scores.items()}

    # Normalize CF and CB scores
    cf_scores_norm = min_max_normalize(cf_scores)
    cb_scores_norm = min_max_normalize(cb_scores)

    print("CF score range AFTER normalization:", min(cf_scores_norm.values()), max(cf_scores_norm.values()))
    print("CB score range AFTER normalization:", min(cb_scores_norm.values()), max(cb_scores_norm.values()))

    # -------------------
    # Mood-based scoring
    # -------------------

    # Each song has a mood column: music_info_metadata['mood_quadrant']
    # Create a simple match score: 1 if same quadrant, else 0
    mood_scores = {}
    for t in set(cf_scores_norm) | set(cb_scores_norm):
        # Access the corresponding one-hot column
        mood_col = f'mood_{user_mood}'
        mood_scores[t] = music_info.loc[music_info['track_id'] == t, mood_col].values[0]

    # Normalize mood scores just like CF/CB
    mood_scores_norm = min_max_normalize(mood_scores)


    # -------------------
    # Hybrid Recommendations
    # -------------------
    
    # Weighted combination (beta_cf + beta_cb + beta_mood = 1)
    beta_cf = 0.2
    beta_cb = 0.2
    beta_mood = 0.6
    
    hybrid_scores = {}
    for t in set(cf_scores_norm) | set(cb_scores_norm):
        hybrid_scores[t] = (
            beta_cf * cf_scores_norm.get(t, 0) +
            beta_cb * cb_scores_norm.get(t, 0) +
            beta_mood * mood_scores_norm.get(t, 0)
        )
    # Sort by hybrid score
    hybrid_sorted = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    hybrid_recommended_tracks = [t for t, s in hybrid_sorted[:n_recommendations]]

    
    # Display recommendations
    print("===========================================================\n")
    print(f"CF Recommendations for user {user_id}:")
    for t in cf_recommended_tracks:
        meta = track_metadata_dict.get(t, {"name": t, "artist": "Unknown"})
        print(f" - {meta['name']} by {meta['artist']}")
    
    print("\n===========================================================\n")
    print(f"CB (kNN) Similar Tracks to '{track_name}':")
    for t in cb_recommended_tracks[:n_recommendations]:
        meta = track_metadata_dict.get(t, {"name": t, "artist": "Unknown"})
        print(f" - {meta['name']} by {meta['artist']}")
    
    print("\n===========================================================\n")
    print(f"Hybrid Recommendations:")
    for t in hybrid_recommended_tracks:
        meta = track_metadata_dict.get(t, {"name": t, "artist": "Unknown"})
        print(f" - {meta['name']} by {meta['artist']}")
    print("===========================================================\n")

    return hybrid_recommended_tracks


# baseline

In [114]:
def recommended_random(user_id,k=10):
    return random.sample(all_items, k)

In [115]:
def reccommended_popularity(user_id,k=10):
    return most_popular_songs[:k]

In [116]:
def recommed_cf(user_id,k=10):
    user_code = cf_user_id_reverse_mapping.get(user_id)
    
    if user_code is None:
        return []
    
    scores = np.dot(user_factors[user_code, :], item_factors.T)
    top_idices = np.argsort(scores)[::-1][:k]
    
    recommed_tracks = [cf_track_id_mapping[i] for i in top_idices]
    return recommed_tracks

In [117]:
def recommend_hybrid(user_id, k=10):
    # kall hybrid uten utskrift og stemning
    hybrid_tracks = recommend_songs_hybrid(
        user_id=user_id,
        track_name=random.choice(music_info_metadata['name']),  # placeholder
        user_item_matrix=user_item_sparse,
        user_factors=user_factors,
        item_factors=item_factors,
        music_info_metadata=music_info_metadata,
        knn_index=knn_index,
        numerical_features=numerical_features,
        user_mood="Q1",  # placeholder
        n_recommendations=k
    )
    return hybrid_tracks


# Hit rate

In [118]:
def evaluate_hit_rate(model_fn, test_data, k=10):
    hits = 0
    total = 0
    
    for user_id, true_items in test_data.items():
        total += 1
        
        recommended = set(model_fn(user_id, k))
        truth = set(true_items)
        if len(recommended & truth) >0:
            hits += 1
    
    hit_rate = hits / total
    return hit_rate

# Evaluation

In [119]:
# for every user
"""k = 10
hit_random = evaluate_hit_rate(recommended_random, user_test_data,k)
hit_pop = evaluate_hit_rate(reccommended_popularity, user_test_data,k)
hit_cf = evaluate_hit_rate(recommed_cf, user_test_data,k)
hit_hybrid = evaluate_hit_rate(recommend_hybrid, user_test_data,k)

print(f"Hit rate for random: {hit_random}")
print(f"Hit rate for popular: {hit_pop}")
print(f"Hit rate for CF: {hit_cf}")
print(f"Hit rate for hybrid: {hit_hybrid}")"""

'k = 10\nhit_random = evaluate_hit_rate(recommended_random, user_test_data,k)\nhit_pop = evaluate_hit_rate(reccommended_popularity, user_test_data,k)\nhit_cf = evaluate_hit_rate(recommed_cf, user_test_data,k)\nhit_hybrid = evaluate_hit_rate(recommend_hybrid, user_test_data,k)\n\nprint(f"Hit rate for random: {hit_random}")\nprint(f"Hit rate for popular: {hit_pop}")\nprint(f"Hit rate for CF: {hit_cf}")\nprint(f"Hit rate for hybrid: {hit_hybrid}")'

In [120]:
# for one random user 
example_user = random.choice(list(user_test_data.keys()))
true_items = set(user_test_data[example_user])

print("random user", example_user)
print("user has actually listned to these tracks:", true_items)

def print_recommendations(model_name,recs,true_item):
    print("\n----------------------------------------------")
    print(model_name)
    print("----------------------------------------------")
    
    for t in recs: 
        track = music_info_metadata[music_info_metadata['track_id'] == t]
        
        if not track.empty:
            name = track['name'].values[0]
            artist = track['artist'].values[0]
        else:
            name = t
            artist = "Unknown"
        hit = " <--- HIT" if t in true_items else ""
        print(f"- {name} ({artist}){hit}")
        

# do 5 for every model for one user 
k = 5 
rec_random = recommended_random(example_user,k)
rec_pop = reccommended_popularity(example_user,k)
rec_cf = recommed_cf(example_user,k)
rec_hybrid = recommend_hybrid(example_user,k)

print_recommendations("random",rec_random,true_items)
print_recommendations("popular",rec_pop,true_items)
print_recommendations("cf",rec_cf,true_items)
print_recommendations("hybrid",rec_hybrid,true_items)
        

random user 941c35db1309822d08033dcb7a515e7f7b699b39
user has actually listned to these tracks: {'TRWRUCY128F424351B', 'TRLKEKF128F9303281', 'TRNYQZQ128F4262D84', 'TRCOFIC128F93173C1', 'TRPFLRB128F14A895D', 'TREUVXG12903CA6D2C', 'TRAECHJ12903CC36DA', 'TRWDNVX128F9313F4D'}
CF score range BEFORE normalization: 0.029138106120753234 0.05591881878310391
CB score range BEFORE normalization: 0.9963807463701079 0.999332816426405
CF score range AFTER normalization: 0.0 1.0
CB score range AFTER normalization: 0.0 1.0

CF Recommendations for user 941c35db1309822d08033dcb7a515e7f7b699b39:
 - Help I'm Alive by Metric
 - Everlasting Light by The Black Keys
 - I'll Be Your Man by The Black Keys
 - Gimme Sympathy by Metric
 - Remember When (Side A) by The Black Keys


CB (kNN) Similar Tracks to 'The Frozen World':
 - Quando o Sol se For by Detonautas Roque Clube
 - Go On, Say It by Blind Pilot
 - Can You Stand The Rain by New Edition
 - Maggie May by Rod Stewart
 - Bye Bye Baby by Bay City Rollers


H

## Hit rate